In [0]:
%sql
DROP SCHEMA IF EXISTS workspace CASCADE;
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
from pyspark.sql.functions import current_timestamp, col

file_path = "/Volumes/workspace/default/superstore_raw/day_1.csv"
df_day_1 = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("escape", "\"") \
    .option("multiLine", "true") \
    .load(file_path)

new_columns = [col.replace(" ", "_").replace("-", "_") for col in df_day_1.columns]
df_day_1 = df_day_1.toDF(*new_columns)

df_bronze = df_day_1.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_file_name", col("_metadata.file_path"))

df_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.orders")

display(spark.table("workspace.bronze.orders"))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit,ingestion_timestamp,source_file_name
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47,2026-06-18T20:03:03.906Z,dbfs:/Volumes/workspace/default/superstore_raw/day_1.csv


In [0]:
# creating a made up amagers table so i can showcase join optimisation later
manager_data = [
    ("South", "Alice"), 
    ("Central", "Bob"), 
    ("East", "Charlie"), 
    ("West", "Diana")
]

managers_df = spark.createDataFrame(manager_data, ["Region", "Regional_Manager"])

managers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.managers")

In [0]:
%sql
CREATE OR REPLACE VIEW silver.orders_deduplicated AS
WITH duplication_check_table AS (
    SELECT *,
           row_number() OVER (PARTITION BY Row_ID ORDER BY ingestion_timestamp DESC) AS duplication_check
    FROM bronze.orders 
)
SELECT * FROM duplication_check_table
WHERE duplication_check = 1;

CREATE OR REPLACE TABLE silver.orders AS 
SELECT 
    Row_ID,
    Order_ID,
    Order_Date,
    Ship_Date,
    UPPER(Ship_Mode) AS Ship_Mode,
    Customer_ID,
    TRIM(Customer_Name) AS Customer_Name,
    Segment,
    Country,
    City,
    State,
    Postal_Code,
    Region,
    Product_ID,
    Category,
    Sub_Category,
    Product_Name,
    CAST (Sales AS NUMERIC(15,4)) AS Sales,
    CAST (Quantity AS INT) AS Quantity,
    CAST (Discount AS NUMERIC(3,2)) AS Discount,
    CAST (Profit AS NUMERIC(15,4)) AS Profit,
    ingestion_timestamp,
    source_file_name
FROM silver.orders_deduplicated
WHERE CAST (Sales AS NUMERIC(15,4)) >= 0 
  AND CAST (Quantity AS INT) > 0
  AND Ship_Date >= Order_Date;

CREATE OR REPLACE TABLE silver.rejected_orders AS 
SELECT * FROM silver.orders_deduplicated
WHERE CAST (Sales AS NUMERIC(15,4)) < 0
   OR CAST (Quantity AS INT) <= 0
   OR Ship_Date < Order_Date;

num_affected_rows,num_inserted_rows


In [0]:
from pyspark.sql.functions import current_timestamp, col

def ingest_data(file_path, view_name):
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("escape", "\"") \
        .option("multiLine", "true") \
        .load(file_path)

    new_columns = [c.replace(" ", "_").replace("-", "_") for c in df.columns]
    df = df.toDF(*new_columns)

    df_bronze = df \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("source_file_name", col("_metadata.file_path"))

    df_bronze.write.format("delta").mode("append").saveAsTable("workspace.bronze.orders")

    df_bronze.createOrReplaceTempView(view_name)
    

In [0]:
file_path = "/Volumes/workspace/default/superstore_raw/day_2.csv"
ingest_data(file_path, "df_day_2")

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW day_2_staging AS
SELECT 
    Row_ID,
    Order_ID,
    Order_Date,
    Ship_Date,
    UPPER(Ship_Mode) AS Ship_Mode,
    Customer_ID,
    TRIM(Customer_Name) AS Customer_Name,
    Segment,
    Country,
    City,
    State,
    Postal_Code,
    Region,
    Product_ID,
    Category,
    Sub_Category,
    Product_Name,
    CAST (Sales AS NUMERIC(15,4)) AS Sales,
    CAST (Quantity AS INT) AS Quantity,
    CAST (Discount AS NUMERIC(3,2)) AS Discount,
    CAST (Profit AS NUMERIC(15,4)) AS Profit,
    ingestion_timestamp,
    source_file_name
FROM df_day_2
WHERE TRY_CAST(Sales AS NUMERIC(15,4)) >= 0 
  AND TRY_CAST(Quantity AS INT) > 0
  AND Ship_Date >= Order_Date;


MERGE INTO silver.orders AS target
USING day_2_staging AS source
ON target.Row_ID = source.Row_ID
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3331,0,0,3331


In [0]:
file_path = "/Volumes/workspace/default/superstore_raw/day_3.csv"
ingest_data(file_path, "df_day_3")

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW day_3_staging AS
SELECT 
    Row_ID,
    Order_ID,
    Order_Date,
    Ship_Date,
    UPPER(Ship_Mode) AS Ship_Mode,
    Customer_ID,
    TRIM(Customer_Name) AS Customer_Name,
    Segment,
    Country,
    City,
    State,
    Postal_Code,
    Region,
    Product_ID,
    Category,
    Sub_Category,
    Product_Name,
    CAST (Sales AS NUMERIC(15,4)) AS Sales,
    CAST (Quantity AS INT) AS Quantity,
    CAST (Discount AS NUMERIC(3,2)) AS Discount,
    CAST (Profit AS NUMERIC(15,4)) AS Profit,
    ingestion_timestamp,
    source_file_name
FROM df_day_3
WHERE TRY_CAST(Sales AS NUMERIC(15,4)) >= 0 
  AND TRY_CAST(Quantity AS INT) > 0
  AND Ship_Date >= Order_Date;


MERGE INTO silver.orders AS target
USING day_3_staging AS source
ON target.Row_ID = source.Row_ID
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3332,0,0,3332


In [0]:
%sql
CREATE OR REPLACE TABLE gold.sales_daily AS 
SELECT 
    DATE(Order_Date) AS Sales_Day,
    SUM(Sales) AS revenue
FROM silver.orders
GROUP BY 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold.sales_region AS
SELECT 
    Region,
    SUM(Sales) AS revenue
FROM silver.orders
GROUP BY 1;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold.sales_category AS
SELECT 
    Category,
    SUM(Sales) AS revenue
FROM silver.orders
GROUP BY 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold.customer_metrics AS
SELECT 
    Customer_ID,
    Customer_Name,
    SUM(Sales) AS revenue,
    COUNT(DISTINCT Order_ID) AS orders
FROM silver.orders
GROUP BY 1, 2;

num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE gold.product_metrics AS
SELECT 
    Product_ID,
    Product_Name,
    Region,
    SUM(Sales) AS revenue,
    COUNT(DISTINCT Order_ID) AS orders
FROM silver.orders
GROUP BY 1, 2, 3;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Top 5 products by revenue

SELECT * 
FROM gold.product_metrics
ORDER BY Revenue DESC
LIMIT 5;

Product_ID,Product_Name,Region,revenue,orders
TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,East,30099.9140,3
TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferencing Unit,South,22638.4800,1
TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Central,17499.9500,1
TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",East,14299.8900,2
TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,Central,14279.9160,3


In [0]:
%sql

-- Top region by revenue
SELECT * 
FROM gold.sales_region
ORDER BY Revenue DESC
LIMIT 1;

Region,revenue
West,725457.8245


In [0]:
%sql

-- Monthly revenue trend
WITH revenue_monthly AS(
SELECT
   YEAR(Sales_Day) AS Sales_Year,
    MONTH(Sales_Day) AS Sales_Month,
    SUM(Revenue) AS Revenue
FROM gold.sales_daily
GROUP BY 1,2
)

SELECT 
    *,
    LAG(Revenue) OVER(ORDER BY Sales_Year, Sales_Month) AS Revenue_Last_Month,
    Revenue - LAG(Revenue) OVER(ORDER BY Sales_Year, Sales_Month) AS Revenue_Diff
FROM revenue_monthly
ORDER BY Sales_Year, Sales_Month;

Sales_Year,Sales_Month,Revenue,Revenue_Last_Month,Revenue_Diff
2014,1,14236.8950,null,null
2014,2,4519.8920,14236.8950,-9717.0030
2014,3,55691.0090,4519.8920,51171.1170
2014,4,28295.3450,55691.0090,-27395.6640
2014,5,23648.2870,28295.3450,-4647.0580
2014,6,34595.1276,23648.2870,10946.8406
2014,7,33946.3930,34595.1276,-648.7346
2014,8,27909.4685,33946.3930,-6036.9245
2014,9,81777.3508,27909.4685,53867.8823
2014,10,31453.3930,81777.3508,-50323.9578


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Running revenue total
SELECT 
    *,
    SUM(Revenue) OVER(ORDER BY Sales_Day) AS Cumulative_Revenue
FROM gold.sales_daily
ORDER BY Sales_Day;

Sales_Day,revenue,Cumulative_Revenue
2014-01-03,16.4480,16.4480
2014-01-04,288.0600,304.5080
2014-01-05,19.5360,324.0440
2014-01-06,4407.1000,4731.1440
2014-01-07,87.1580,4818.3020
2014-01-09,40.5440,4858.8460
2014-01-10,54.8300,4913.6760
2014-01-11,9.9400,4923.6160
2014-01-13,3553.7950,8477.4110
2014-01-14,61.9600,8539.3710


In [0]:

%sql
-- Top product per region
WITH products_revenue_in_region AS (
SELECT 
    Region,
    Product_ID,
    Product_Name,
    SUM(Sales) AS Revenue
FROM silver.orders
GROUP BY 1,2,3
),

product_ranks_in_region AS (
SELECT 
    *,
    ROW_NUMBER() OVER(PARTITION BY Region ORDER BY Revenue DESC) AS product_rank
FROM products_revenue_in_region
)

SELECT * FROM product_ranks_in_region
WHERE product_rank = 1;


Region,Product_ID,Product_Name,Revenue,product_rank
Central,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,17499.9500,1
East,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,30099.9140,1
South,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferencing Unit,22638.4800,1
West,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,13999.9600,1


In [0]:
%sql
-- join without optimisation
-- unfortuanetly my sample table is very small that auto broadcast join is triggered even without hints or anything
-- if this was cluster I could have SET spark.sql.autoBroadcastJoinThreshold = -1; but I can't on serverless
-- so i am using a bad hint to show the difference

EXPLAIN 
SELECT /*+ MERGE(m) */
    o.*,  
    m.Regional_Manager
FROM silver.orders o
LEFT JOIN silver.managers m 
    ON o.Region = m.Region;

plan
"== Physical Plan == AdaptiveSparkPlan isFinalPlan=false +- == Initial Plan == Project [Row_ID#18417, Order_ID#18418, Order_Date#18419, Ship_Date#18420, Ship_Mode#18421, Customer_ID#18422, Customer_Name#18423, Segment#18424, Country#18425, City#18426, State#18427, Postal_Code#18428, Region#18429, Product_ID#18430, Category#18431, Sub_Category#18432, Product_Name#18433, Sales#18434, Quantity#18435, Discount#18436, Profit#18437, ingestion_timestamp#18438, source_file_name#18439, Regional_Manager#18443] +- SortMergeJoin [Region#18429], [Region#18442], LeftOuter :- ColumnarToRow : +- PhotonResultStage : +- PhotonSort [Region#18429 ASC NULLS FIRST] : +- PhotonShuffleExchangeSource : +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#10070] : +- PhotonShuffleExchangeSink hashpartitioning(Region#18429, 16) : +- PhotonScan parquet workspace.silver.orders[Row_ID#18417,Order_ID#18418,Order_Date#18419,Ship_Date#18420,Ship_Mode#18421,Customer_ID#18422,Customer_Name#18423,Segment#18424,Country#18425,City#18426,State#18427,Postal_Code#18428,Region#18429,Product_ID#18430,Category#18431,Sub_Category#18432,Product_Name#18433,Sales#18434,Quantity#18435,Discount#18436,Profit#18437,ingestion_timestamp#18438,source_file_name#18439] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-3ril2/uc/ba425e00-629b-43cf-bc03-ae5382a91c60..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<Row_ID:int,Order_ID:string,Order_Date:date,Ship_Date:date,Ship_Mode:string,Customer_ID:str..., RequiredDataFilters: [] +- ColumnarToRow +- PhotonResultStage +- PhotonSort [Region#18442 ASC NULLS FIRST] +- PhotonShuffleExchangeSource +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#10078] +- PhotonShuffleExchangeSink hashpartitioning(Region#18442, 16) +- PhotonScan parquet workspace.silver.managers[Region#18442,Regional_Manager#18443] DataFilters: [isnotnull(Region#18442)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-3ril2/uc/ba425e00-629b-43cf-bc03-ae5382a91c60..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct, RequiredDataFilters: [isnotnull(Region#18442)] == Photon Explanation == Photon does not fully support the query because: Unsupported node: SortMergeJoin [Region#18429], [Region#18442], LeftOuter. Reference node: SortMergeJoin [Region#18429], [Region#18442], LeftOuter == Optimizer Statistics (table names per statistics state) == missing = partial = full = managers, orders"


In [0]:
%sql
-- join with hitn to broadcast and as result optimisation
EXPLAIN
SELECT /*+ BROADCAST(m) */ 
    o.*, 
    m.Regional_Manager
FROM silver.orders o
LEFT JOIN silver.managers m 
    ON o.Region = m.Region;

plan
"== Physical Plan == AdaptiveSparkPlan isFinalPlan=false +- == Initial Plan == PhotonResultStage +- PhotonColumnarToRow +- PhotonProject [Row_ID#18497, Order_ID#18498, Order_Date#18499, Ship_Date#18500, Ship_Mode#18501, Customer_ID#18502, Customer_Name#18503, Segment#18504, Country#18505, City#18506, State#18507, Postal_Code#18508, Region#18509, Product_ID#18510, Category#18511, Sub_Category#18512, Product_Name#18513, Sales#18514, Quantity#18515, Discount#18516, Profit#18517, ingestion_timestamp#18518, source_file_name#18519, Regional_Manager#18523] +- PhotonBroadcastHashJoin [Region#18509], [Region#18522], LeftOuter, BuildRight, false, true :- PhotonScan parquet workspace.silver.orders[Row_ID#18497,Order_ID#18498,Order_Date#18499,Ship_Date#18500,Ship_Mode#18501,Customer_ID#18502,Customer_Name#18503,Segment#18504,Country#18505,City#18506,State#18507,Postal_Code#18508,Region#18509,Product_ID#18510,Category#18511,Sub_Category#18512,Product_Name#18513,Sales#18514,Quantity#18515,Discount#18516,Profit#18517,ingestion_timestamp#18518,source_file_name#18519] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-3ril2/uc/ba425e00-629b-43cf-bc03-ae5382a91c60..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<Row_ID:int,Order_ID:string,Order_Date:date,Ship_Date:date,Ship_Mode:string,Customer_ID:str..., RequiredDataFilters: [] +- PhotonShuffleExchangeSource +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#10164] +- PhotonShuffleExchangeSink SinglePartition +- PhotonScan parquet workspace.silver.managers[Region#18522,Regional_Manager#18523] DataFilters: [isnotnull(Region#18522)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-3ril2/uc/ba425e00-629b-43cf-bc03-ae5382a91c60..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct, RequiredDataFilters: [isnotnull(Region#18522)] == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = managers, orders"


In [0]:
%sql
SELECT * 
FROM gold.sales_region
ORDER BY Revenue DESC;

Region,revenue
West,725457.8245
East,678781.2400
Central,501239.8908
South,391721.9050


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT * 
FROM gold.sales_category
ORDER BY Revenue DESC;

Category,revenue
Technology,836154.0330
Furniture,741999.7953
Office Supplies,719047.0320


Databricks visualization. Run in Databricks to view.